## 04- Analyse Multivariée

In [41]:
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib as plt
import sklearn  as skl 
df = pd.read_csv("../Data/Data_brut/Customer-Churn-Records.csv")

##### 1- Création des nouvelles variables à partir des variables les plus discriminantes

In [42]:
df["Risque_Nbproduit"] = (df["NumOfProducts"] >= 3).astype(int)
df["Client_Allemangne"] = (df["Geography"] == "Germany").astype(int)
df["Solde_Positif"] = (df["Balance"] > 0).astype(int)
df["Client_Senior"] = (df["Age"] >= 50).astype(int)
df["Clt_Inact_Monopdt"] = ((df["IsActiveMember"] == 0) & (df["NumOfProducts"] == 1)).astype(int)
df["appréciation_Nbproduit"] = df["NumOfProducts"].map({
    1: "risque",      
    2: "fidele",       
    3: "critique",    
    4: "critique"    
})

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   RowNumber               10000 non-null  int64  
 1   CustomerId              10000 non-null  int64  
 2   Surname                 10000 non-null  object 
 3   CreditScore             10000 non-null  int64  
 4   Geography               10000 non-null  object 
 5   Gender                  10000 non-null  object 
 6   Age                     10000 non-null  int64  
 7   Tenure                  10000 non-null  int64  
 8   Balance                 10000 non-null  float64
 9   NumOfProducts           10000 non-null  int64  
 10  HasCrCard               10000 non-null  int64  
 11  IsActiveMember          10000 non-null  int64  
 12  EstimatedSalary         10000 non-null  float64
 13  Exited                  10000 non-null  int64  
 14  Complain                10000 non-null 

##### 2- Encodage des variables catégorielles 

In [44]:
df = pd.get_dummies(df, columns=["Geography"],     drop_first=False, dtype = int)
df = pd.get_dummies(df, columns=["appréciation_Nbproduit"],drop_first=False, dtype = int)# one-hote encoding
df["Gender"] = (df["Gender"] == "Female").astype(int)# label encoding

##### 3- Suppression des variables faiblements associées à Exited

In [45]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname", "Tenure", "HasCrCard", "EstimatedSalary", "Complain", "Satisfaction Score", "Card Type", "Point Earned"])

##### 4- Normalisation des variables (Age et Balance) : pour qu'elles soient à la même échelle que les autres

In [46]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[["Age", "Balance"]] = scaler.fit_transform(df[["Age", "Balance"]])

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   CreditScore                      10000 non-null  int64  
 1   Gender                           10000 non-null  int64  
 2   Age                              10000 non-null  float64
 3   Balance                          10000 non-null  float64
 4   NumOfProducts                    10000 non-null  int64  
 5   IsActiveMember                   10000 non-null  int64  
 6   Exited                           10000 non-null  int64  
 7   Risque_Nbproduit                 10000 non-null  int64  
 8   Client_Allemangne                10000 non-null  int64  
 9   Solde_Positif                    10000 non-null  int64  
 10  Client_Senior                    10000 non-null  int64  
 11  Clt_Inact_Monopdt                10000 non-null  int64  
 12  Geography_France   

In [48]:
df = df.drop(columns=["CreditScore", "NumOfProducts"])

##### 5- Analyse de la multicolinéarité des variables explicatives 

In [49]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
X = df[["Age","Gender", "Balance", "IsActiveMember", "Risque_Nbproduit", "Client_Allemangne", 
        "Solde_Positif","Client_Senior", "Clt_Inact_Monopdt","Geography_France", "Geography_Germany", 
        "Geography_Spain", "appréciation_Nbproduit_critique", "appréciation_Nbproduit_fidele",
        "appréciation_Nbproduit_risque"
        
        ]]
X = sm.add_constant(X)

vif = pd.DataFrame()
vif["Variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif = vif[vif["Variable"] != "const"]

print(vif)
print("=== Seuils VIF ===")
print(f"VIF < 5  -> {vif[vif['VIF'] < 5]['Variable'].tolist()}")
print(f"VIF 5-10 -> {vif[(vif['VIF'] >= 5) & (vif['VIF'] <= 10)]['Variable'].tolist()}")
print(f"VIF > 10 -> {vif[vif['VIF'] > 10]['Variable'].tolist()}")
print(f"VIF < 5  -> seuil Acceptable ")
print(f"VIF 5-10 -> seuil Modéré")
print(f"VIF > 10  ->  seuil élevé  ")

                           Variable       VIF
1                               Age  2.361273
2                            Gender  1.004422
3                           Balance  6.742563
4                    IsActiveMember  2.065423
5                  Risque_Nbproduit       inf
6                 Client_Allemangne       inf
7                     Solde_Positif  7.245570
8                     Client_Senior  2.362278
9                 Clt_Inact_Monopdt  3.038340
10                 Geography_France       inf
11                Geography_Germany       inf
12                  Geography_Spain       inf
13  appréciation_Nbproduit_critique       inf
14    appréciation_Nbproduit_fidele       inf
15    appréciation_Nbproduit_risque       inf
=== Seuils VIF ===
VIF < 5  -> ['Age', 'Gender', 'IsActiveMember', 'Client_Senior', 'Clt_Inact_Monopdt']
VIF 5-10 -> ['Balance', 'Solde_Positif']
VIF > 10 -> ['Risque_Nbproduit', 'Client_Allemangne', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'app

c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


##### - Suppression des variables correlées 

In [50]:
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib as plt
import sklearn  as skl 
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
df = df.drop(columns=["appréciation_Nbproduit_critique", "appréciation_Nbproduit_fidele", "appréciation_Nbproduit_risque", "Geography_France","Balance", "Geography_Germany"])

In [51]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
X = df[["Age","Gender",  "IsActiveMember", "Risque_Nbproduit", "Client_Allemangne", 
        "Solde_Positif","Client_Senior", "Clt_Inact_Monopdt", 
        "Geography_Spain"
        
        ]]
X = sm.add_constant(X)

vif = pd.DataFrame()
vif["Variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif = vif[vif["Variable"] != "const"]

print(vif)
print("=== Seuils VIF ===")
print(f"VIF < 5  -> {vif[vif['VIF'] < 5]['Variable'].tolist()}")
print(f"VIF 5-10 -> {vif[(vif['VIF'] >= 5) & (vif['VIF'] <= 10)]['Variable'].tolist()}")
print(f"VIF > 10 -> {vif[vif['VIF'] > 10]['Variable'].tolist()}")
print(f"VIF < 5  -> seuil Acceptable ")
print(f"VIF 5-10 -> seuil Modéré")
print(f"VIF > 10  ->  seuil élevé  ")

            Variable       VIF
1                Age  2.361039
2             Gender  1.003918
3     IsActiveMember  1.659190
4   Risque_Nbproduit  1.041420
5  Client_Allemangne  1.373007
6      Solde_Positif  1.331117
7      Client_Senior  2.361340
8  Clt_Inact_Monopdt  1.739201
9    Geography_Spain  1.125292
=== Seuils VIF ===
VIF < 5  -> ['Age', 'Gender', 'IsActiveMember', 'Risque_Nbproduit', 'Client_Allemangne', 'Solde_Positif', 'Client_Senior', 'Clt_Inact_Monopdt', 'Geography_Spain']
VIF 5-10 -> []
VIF > 10 -> []
VIF < 5  -> seuil Acceptable 
VIF 5-10 -> seuil Modéré
VIF > 10  ->  seuil élevé  


In [52]:
df.head()

,Gender,Age,IsActiveMember,Exited,Risque_Nbproduit,Client_Allemangne,Solde_Positif,Client_Senior,Clt_Inact_Monopdt,Geography_Spain
0,1,0.293517,1,1,0,0,0,0,0,0
1,1,0.198164,1,0,0,0,1,0,0,1
2,1,0.293517,0,1,1,0,1,0,0,0
3,1,0.007457,0,0,0,0,0,0,0,0
4,1,0.388871,1,0,0,0,1,0,0,1


##### - Exportation de la base finale (pour la prédiction du départ des clients)

In [53]:
df.to_csv("../Data/Data_for_prediction/Data_Churn_Prediction.csv", index=False)

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             10000 non-null  int64  
 1   Age                10000 non-null  float64
 2   IsActiveMember     10000 non-null  int64  
 3   Exited             10000 non-null  int64  
 4   Risque_Nbproduit   10000 non-null  int64  
 5   Client_Allemangne  10000 non-null  int64  
 6   Solde_Positif      10000 non-null  int64  
 7   Client_Senior      10000 non-null  int64  
 8   Clt_Inact_Monopdt  10000 non-null  int64  
 9   Geography_Spain    10000 non-null  int64  
dtypes: float64(1), int64(9)
memory usage: 781.4 KB
